## Asyncio & Non-Blocking Event Loops

Async programming is essential for high-throughput model serving, distributed RL inference servers, streaming token responses, and handling tens of thousands of concurrent I/O connections on a single OS thread.

Async programming lets one program start something that takes time, and while waiting, work on something else instead of sitting idle.

Don't think:

Async = faster CPU computation.

Think:

Async = don't waste time sitting idle while waiting for I/O.

I/O includes things like:

network requests
database queries
reading from sockets
waiting for another server
streaming data

2. The Core Building Blocks
Coroutines (async def): Functions defined with async def. Calling one returns a coroutine object (it does not execute immediately).

The await Expression: Passes control back to the event loop. Can only be used inside an async def function, and can only await:

Other coroutines.

Tasks (asyncio.Task).

Futures (asyncio.Future).

Tasks (asyncio.create_task(coro)): Wraps a coroutine into a Task object and schedules it immediately onto the event loop to run concurrently in the background.

Gathering (asyncio.gather(*tasks)): Runs multiple awaitables concurrently and aggregates their return values in order.

Python

In [1]:
import aiohttp
import asyncio

async def hello():
    print("Hello")
    await asyncio.sleep(2)
    print("Hello")
asyncio.run(hello())
#->> in .py files

RuntimeError: asyncio.run() cannot be called from a running event loop

In [2]:
import aiohttp
import asyncio

async def hello():
    print("Hello")
    await asyncio.sleep(2)
    print("Hello")
await hello()
#->> in .py files

Hello
Hello


In [5]:
import asyncio
import time

async def model_request(request_id):
    print(f"request for {request_id} started")
    
    await asyncio.sleep(2)
    
    print(f"req for {request_id} done")
    return(f"{request_id} resource")

async def main():
    print("starting model request")
    start = time.perf_counter()
    results = await asyncio.gather(
        model_request(1),
        model_request(2),
        model_request(3)
    )
    end = time.perf_counter()
    print(results)
    print("total time",end-start)
await main()
    

starting model request
request for 1 started
request for 2 started
request for 3 started
req for 1 done
req for 2 done
req for 3 done
['1 resource', '2 resource', '3 resource']
total time 2.0065997000056086


asyncio is about concurrency, not magic parallel CPU execution

Exercise: now the important part

Don't use asyncio.gather().

Build this yourself:

In [16]:
import asyncio
import time

async def request(req_id):
    print("request for ", req_id, " started")
    
    await asyncio.sleep(2)
    
    print("request for ",req_id," done")
    return req_id

async def main():
    
    print("processing  req starts")
    
    processes = []
    results = []
    
    start = time.perf_counter()
    
    for i in range(4):
        processes.append(asyncio.create_task(request(i)))
    
    results = [await process for process in processes]
    # inbuilt->> results = await asyncio.gather(*processes)
    end = time.perf_counter()
    print(results)
    print("time taken ",end-start)

In [17]:
await main()

processing  req starts
request for  0  started
request for  1  started
request for  2  started
request for  3  started
request for  0  done
request for  2  done
request for  1  done
request for  3  done
[0, 1, 2, 3]
time taken  2.0135487000079593


In [22]:
def cpu(n):
    total = 0
    for i in range(n):
        total+=i*i
        
async def work(req_id):
    print("req_id ",req_id, " started")
    N = 20_000_000
    start = time.perf_counter()
    result = cpu(N)
    print("returning results ",req_id, "time ", time.perf_counter()-start)
    return result

async def main():
    
    start = time.perf_counter()
    
    processes = []
    for i in range(4):
        processes.append(asyncio.create_task(work(i)))
    
    results = await asyncio.gather(*processes)
    
    print("time total ",time.perf_counter()-start)

await main()

req_id  0  started
returning results  0 time  1.5325628999999026
req_id  1  started
returning results  1 time  1.3746029999892926
req_id  2  started
returning results  2 time  1.2490087000041967
req_id  3  started
returning results  3 time  1.0908319000009215
time total  5.2497761999984505


## CPU bound task with await slows it down more

In [25]:
async def cpu_work(n):
    total = 0

    for i in range(n):
        total += i * i

    return total
async def main():
    start = time.perf_counter()

    tasks = [
        asyncio.create_task(cpu_work(20_000_000))
        for _ in range(4)
    ]

    results = await asyncio.gather(*tasks)

    print("Time:", time.perf_counter() - start)

await main()

Time: 4.365238999991561


In [24]:
async def cpu_work(n):
    total = 0

    for i in range(n):
        total += i * i

        if i % 1_000_000 == 0:
            await asyncio.sleep(0)

    return total
async def main():
    start = time.perf_counter()

    tasks = [
        asyncio.create_task(cpu_work(20_000_000))
        for _ in range(4)
    ]

    results = await asyncio.gather(*tasks)

    print("Time:", time.perf_counter() - start)

await main()

Time: 7.050839300005464


## with server side limits:

In [29]:
import asyncio
import aiohttp
import time

async def fetch(session, req_id):
    url = "https://httpbin.org/delay/2"

    start = time.perf_counter()

    async with session.get(url) as response:
        await response.text()

    print(f"Request {req_id} done: {time.perf_counter() - start:.2f}s")


async def main():
    async with aiohttp.ClientSession() as session:

        tasks = [
            asyncio.create_task(fetch(session, i))
            for i in range(20)
        ]

        await asyncio.gather(*tasks)


await main()

Request 18 done: 1.89s
Request 14 done: 1.89s
Request 16 done: 1.89s
Request 11 done: 1.89s
Request 17 done: 1.89s
Request 6 done: 1.92s
Request 4 done: 1.92s
Request 13 done: 1.92s
Request 2 done: 1.92s
Request 15 done: 1.92s
Request 0 done: 1.92s
Request 9 done: 1.92s
Request 3 done: 1.92s
Request 7 done: 1.92s
Request 19 done: 1.92s
Request 12 done: 1.92s
Request 8 done: 1.93s
Request 10 done: 1.93s
Request 5 done: 1.93s
Request 1 done: 1.93s


One very important distinction

The semaphore doesn't make the requests faster.

It controls how much concurrency you're allowing.

That's extremely useful in an inference server because your GPU might handle:
        10,000 clients
              ↓
       Async event loop
              ↓
      ┌───────────────┐
      │ Semaphore(5)  │
      └───────────────┘
              ↓
       5 active jobs
              ↓
           GPU/API

In [27]:
import asyncio
import aiohttp
import time

# Maximum 5 requests can be inside the HTTP section at once
semaphore = asyncio.Semaphore(5)


async def fetch(session, req_id):
    print(f"Request {req_id} waiting...")

    # Wait here until one of the 5 slots becomes available
    async with semaphore:
        print(f"Request {req_id} started")

        start = time.perf_counter()

        url = "https://httpbin.org/delay/2"

        async with session.get(url) as response:
            await response.text()

        elapsed = time.perf_counter() - start

        print(f"Request {req_id} done: {elapsed:.2f}s")


async def main():

    start = time.perf_counter()

    async with aiohttp.ClientSession() as session:

        # Create 20 tasks
        tasks = [
            asyncio.create_task(fetch(session, i))
            for i in range(20)
        ]

        # Wait for all 20
        await asyncio.gather(*tasks)

    total = time.perf_counter() - start

    print(f"\nTotal time: {total:.2f}s")


await main()

Request 0 waiting...
Request 0 started
Request 1 waiting...
Request 1 started
Request 2 waiting...
Request 2 started
Request 3 waiting...
Request 3 started
Request 4 waiting...
Request 4 started
Request 5 waiting...
Request 6 waiting...
Request 7 waiting...
Request 8 waiting...
Request 9 waiting...
Request 10 waiting...
Request 11 waiting...
Request 12 waiting...
Request 13 waiting...
Request 14 waiting...
Request 15 waiting...
Request 16 waiting...
Request 17 waiting...
Request 18 waiting...
Request 19 waiting...
Request 4 done: 1.75s
Request 5 started
Request 0 done: 1.83s
Request 6 started
Request 3 done: 1.83s
Request 7 started
Request 1 done: 1.84s
Request 8 started
Request 2 done: 1.85s
Request 9 started
Request 5 done: 0.24s
Request 10 started
Request 6 done: 0.26s
Request 11 started
Request 7 done: 0.27s
Request 12 started
Request 8 done: 0.27s
Request 13 started
Request 9 done: 0.27s
Request 14 started
Request 10 done: 0.24s
Request 15 started
Request 11 done: 0.27s
Request 16

Queue + Semaphore

Imagine your server receives 100 requests, but your GPU can comfortably process only 5 at a time.

You don't want to reject everything immediately, and you don't want to run all 100 simultaneously.

In [30]:
import asyncio
import random
import time

queue = asyncio.Queue()


async def producer():
    # Simulate requests arriving
    for request_id in range(20):
        await queue.put(request_id)
        print(f"Request {request_id} entered queue")

        await asyncio.sleep(0.1)


async def worker(worker_id):
    while True:
        request_id = await queue.get()

        print(
            f"Worker {worker_id} processing "
            f"request {request_id}"
        )

        # Simulate model inference
        await asyncio.sleep(random.uniform(1, 2))

        print(
            f"Worker {worker_id} finished "
            f"request {request_id}"
        )

        queue.task_done()


async def main():

    # Start 5 workers
    workers = [
        asyncio.create_task(worker(i))
        for i in range(5)
    ]

    # Start receiving requests
    await producer()

    # Wait until every queued request is processed
    await queue.join()

    # Stop workers
    for worker_task in workers:
        worker_task.cancel()

    await asyncio.gather(
        *workers,
        return_exceptions=True
    )


await main()

Request 0 entered queue
Worker 0 processing request 0
Request 1 entered queue
Worker 1 processing request 1
Request 2 entered queue
Worker 2 processing request 2
Request 3 entered queue
Worker 3 processing request 3
Request 4 entered queue
Worker 4 processing request 4
Request 5 entered queue
Request 6 entered queue
Request 7 entered queue
Request 8 entered queue
Request 9 entered queue
Worker 0 finished request 0
Worker 0 processing request 5
Request 10 entered queue
Request 11 entered queue
Request 12 entered queue
Worker 1 finished request 1
Worker 1 processing request 6
Request 13 entered queue
Request 14 entered queue
Request 15 entered queue
Request 16 entered queue
Worker 3 finished request 3
Worker 3 processing request 7
Request 17 entered queue
Request 18 entered queue
Worker 2 finished request 2
Worker 2 processing request 8
Request 19 entered queue
Worker 4 finished request 4
Worker 4 processing request 9
Worker 0 finished request 5
Worker 0 processing request 10
Worker 1 fi

Semaphore:

"At most N operations can happen concurrently."

Queue:

"Requests that can't run yet wait here."

Backpressure:

"Don't let incoming work overwhelm the system."

# STREAMING

In [36]:
import asyncio

async def generate_tokens():
    tokens = [
        "ai",
        "is",
        "very",
        "dumb",
        "brohh"
    ]
    
    for token in tokens:
        await asyncio.sleep(0.5)
        yield token
async def main():
    async for token in generate_tokens():
        print(token,end=" ",flush = True)

await main()

ai is very dumb brohh 

Why async for?

A normal generator:

for token in generate_tokens():

assumes the next item is available now.

We used:

async def

and:

yield

Together they create an async generator.

### GPU sim

In [39]:
semaphore = asyncio.Semaphore(2)
async def gpu(req_id):
    
    async with semaphore:
        print(f"GPU running req id ",req_id)
        
        await asyncio.sleep(random.uniform(2,4))
        
        print(f"GPU finished req id ",req_id)
        
        return f"result for req id {req_id}"
    
async def client(req_id):
    
    print(f"client connnected with req id {req_id}")
    
    await gpu(req_id)
    
    print(f"cleint recevied freom gpu {req_id}")

async def main():
    clients = [asyncio.create_task(client(i)) for i in range(8)]
    
    response = await asyncio.gather(*clients)
    
await main()

client connnected with req id 0
GPU running req id  0
client connnected with req id 1
GPU running req id  1
client connnected with req id 2
client connnected with req id 3
client connnected with req id 4
client connnected with req id 5
client connnected with req id 6
client connnected with req id 7
GPU finished req id  0
cleint recevied freom gpu 0
GPU running req id  2
GPU finished req id  1
cleint recevied freom gpu 1
GPU running req id  3
GPU finished req id  3
cleint recevied freom gpu 3
GPU running req id  4
GPU finished req id  2
cleint recevied freom gpu 2
GPU running req id  5
GPU finished req id  4
cleint recevied freom gpu 4
GPU running req id  6
GPU finished req id  5
cleint recevied freom gpu 5
GPU running req id  7
GPU finished req id  6
cleint recevied freom gpu 6
GPU finished req id  7
cleint recevied freom gpu 7
